# Quizzer Notebook

Generate grounded quizzes, answer keys, and an audit from explicitly selected course files. Run the cells from top to bottom. The generation cell calls the configured LLM API and writes files under `outputs/notebook_run/`.

In [1]:
from pathlib import Path
from quizzer import generate_quizzes_from_files

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Start Jupyter from the Quizzer project root before running this notebook.")

PROJECT_ROOT

PosixPath('/Users/jonathanma/Desktop/Projects/quizzer')

## Configure the quiz

Use `mixed` for a mix of multiple-choice and multiple-select questions, or `open` for open-ended questions. Every file listed below is used; topic names are derived from the lecture-note filenames.

In [2]:
NUM_VERSIONS = 1
NUM_QUESTIONS = 15
QUESTION_TYPE = "mixed"  # "mixed" or "open"
OUTPUT_FORMAT = "markdown"  # "markdown" or "tex"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_run"

## Choose input files

Paths may point anywhere accessible to the notebook. Add or remove files here to control the material used for generation; no separate topic filter is required.

In [3]:
INPUT_FILES = {
    "syllabus": PROJECT_ROOT / "inputs" / "syllabus" / "syllabus.md",
    "lecture_notes": [
        PROJECT_ROOT / "inputs" / "lecture_notes" / "01-Desc.html",
        PROJECT_ROOT / "inputs" / "lecture_notes" / "01-Intro.pdf",
    ],
    "learning_outcomes": [
        PROJECT_ROOT / "inputs" / "MLO" / "01_Desc.md"    
    ],
}

for group, files in INPUT_FILES.items():
    paths = files if isinstance(files, list) else [files]
    for path in paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f"Missing {group} file: {path}")

INPUT_FILES

{'syllabus': PosixPath('/Users/jonathanma/Desktop/Projects/quizzer/inputs/syllabus/syllabus.md'),
 'lecture_notes': [PosixPath('/Users/jonathanma/Desktop/Projects/quizzer/inputs/lecture_notes/01-Desc.html'),
  PosixPath('/Users/jonathanma/Desktop/Projects/quizzer/inputs/lecture_notes/01-Intro.pdf')],
 'learning_outcomes': [PosixPath('/Users/jonathanma/Desktop/Projects/quizzer/inputs/MLO/01_Desc.md')]}

## Generate

Running this cell makes the LLM requests. Ensure `OPENAI_API_KEY` is available in your environment or project `.env` file.

In [4]:
manifest = generate_quizzes_from_files(
    input_files=INPUT_FILES,
    output_dir=OUTPUT_DIR,
    num_versions=NUM_VERSIONS,
    num_questions=NUM_QUESTIONS,
    question_type=QUESTION_TYPE,
    output_format=OUTPUT_FORMAT,
)

manifest

{'quizzes': ['notebook_run/quizzes/quiz_01.md'],
 'answer_keys': ['notebook_run/answer_keys/quiz_01_key.md'],
 'supplementary': ['notebook_run/supplementary/code/quiz_01_q04_plot.py',
  'notebook_run/supplementary/plots/quiz_01_q04_plot.png',
  'notebook_run/supplementary/code/quiz_01_q08_plot.py',
  'notebook_run/supplementary/plots/quiz_01_q08_plot.png',
  'notebook_run/supplementary/code/quiz_01_q12_plot.py',
  'notebook_run/supplementary/plots/quiz_01_q12_plot.png'],
 'audit': 'notebook_run/audit/generation_audit.md',
 'blueprint_file': 'notebook_run/audit/blueprint.json',
 'blueprint': {'num_versions': 1,
  'num_questions': 15,
  'question_style': 'mixed',
  'topics': ['Desc', 'Intro'],
  'slots': [{'number': 1,
    'topic': 'Desc',
    'learning_outcome': {'identifier': 'LO-1',
     'statement': 'Python and Jupyter Notebook support interactive data analysis with code, text, equations, and visualizations.',
     'category': 'Key Concepts',
     'source': 'MLO/01_Desc.md'},
    'qu

## Inspect the blueprint

In [5]:
import json

print(json.dumps(manifest["blueprint"], indent=2))

{
  "num_versions": 1,
  "num_questions": 15,
  "question_style": "mixed",
  "topics": [
    "Desc",
    "Intro"
  ],
  "slots": [
    {
      "number": 1,
      "topic": "Desc",
      "learning_outcome": {
        "identifier": "LO-1",
        "statement": "Python and Jupyter Notebook support interactive data analysis with code, text, equations, and visualizations.",
        "category": "Key Concepts",
        "source": "MLO/01_Desc.md"
      },
      "question_kind": "single_choice",
      "difficulty": "foundational",
      "modality": "conceptual"
    },
    {
      "number": 2,
      "topic": "Intro",
      "learning_outcome": {
        "identifier": "LO-2",
        "statement": "Pandas loads and manipulates tabular data; NumPy supports numerical arrays; Matplotlib supports visualization.",
        "category": "Key Concepts",
        "source": "MLO/01_Desc.md"
      },
      "question_kind": "multiple_select",
      "difficulty": "intermediate",
      "modality": "formula"
    },


## Preview generated files

In [6]:
from IPython.display import Markdown, display

quiz_files = sorted((OUTPUT_DIR / "quizzes").glob("*"))
answer_key_files = sorted((OUTPUT_DIR / "answer_keys").glob("*"))
audit_file = OUTPUT_DIR / "audit" / "generation_audit.md"

print("Quiz files:", *quiz_files, sep="\n- " )
print("\nAnswer keys:", *answer_key_files, sep="\n- " )

if OUTPUT_FORMAT == "markdown" and quiz_files:
    display(Markdown(quiz_files[0].read_text(encoding="utf-8")))

display(Markdown(audit_file.read_text(encoding="utf-8")))

Quiz files:
- /Users/jonathanma/Desktop/Projects/quizzer/outputs/notebook_run/quizzes/quiz_01.md

Answer keys:
- /Users/jonathanma/Desktop/Projects/quizzer/outputs/notebook_run/answer_keys/quiz_01_key.md


# Quiz

1. What role does Jupyter Notebook play in data analysis?

A. It supports interactive data analysis with code, text, equations, and visualizations.  
B. It compiles Python code into an executable file.  
C. It provides a database for storing large datasets.  
D. It is only used for statistical calculations.  

2. Which of the following operations can be performed using Pandas and NumPy? (Select all that apply)

A. Load and manipulate tabular data  
B. Support numerical arrays  
C. Generate 3D plots  
D. Compute statistical measures  

3. What will be the output of the following code snippet?

```python
import numpy as np
N = 5
x = np.array([1, 2, 3, 4, 5])
print(x[1:3])
```

A. [2, 3]  
B. [1, 2]  
C. [3, 4]  
D. [2, 3, 4]  

![Signal plus Noise Representation](../supplementary/plots/quiz_01_q04_plot.png)

4. Given the following model where observations are represented as a signal plus random noise, identify the meaning of $Y_i=f(X_i)+\epsilon_i$ in the context of data science:
- $Y_i$ = output variable,
- $f(X_i)$ = model function applied to input $X_i$,
- $\epsilon_i$ = random noise.

A. The relationship between input and predicted output.  
B. An illustration of a completely deterministic process.  
C. A statistical representation of regression without noise.  
D. The distribution of variables in a linear model.  

5. Which of the following tasks are included in statistical learning? (Select all that apply)

A. Classification  
B. Regression  
C. Clustering  
D. Data organization  

6. What is the formula for calculating the sample mean?

A. $\bar{x} = \frac{1}{N}\sum_{i=1}^N x_i$  
B. $\bar{x} = \frac{1}{N-1}\sum_{i=1}^N x_i$  
C. $\bar{x} = \frac{1}{N}\sum_{i=0}^{N} x_i$  
D. $\bar{x} = \sum_{i=1}^N x_i$  

7. What output will the following code produce?\n\n```python\nimport numpy as np\n# Create an array of integers from 1 to 5\narr = np.array([1, 2, 3, 4, 5])\n# Calculate the mean\nmean_value = np.mean(arr)\nprint(mean_value)\n```

A. 3  
B. 2  
C. 4  
D. 15  

![Sample Histogram](../supplementary/plots/quiz_01_q08_plot.png)

8. Based on the histogram provided, which of the following statements are true? (Select all that apply)

A. The distribution has a single peak.  
B. The distribution is symmetric.  
C. There are multiple peaks present.  
D. The distribution is skewed to the right.  

9. What does excess kurtosis indicate about a distribution?

A. The distribution has a higher peak and fatter tails compared to a normal distribution.  
B. The distribution is uniformly distributed.  
C. The distribution is perfectly symmetrical.  
D. The distribution has lighter tails than a normal distribution.  

10. What is the formula for calculating sample variance?

A. $s^2 = \frac{1}{N}\sum_{i=1}^N (x_i - \bar{x})^2$  
B. $s^2 = \frac{1}{N-1}\sum_{i=1}^N (x_i - \bar{x})^2$  
C. $s^2 = \sum_{i=1}^N (x_i - \bar{x})^2$  
D. $s^2 = \frac{1}{N} \left(\sum_{i=1}^N x_i \right)^2$  

11. Which of the following Python libraries are generally used for data manipulation, numerical computing, and data visualization? Select all that apply.

A. Pandas  
B. NumPy  
C. Matplotlib  
D. Scikit-learn  

![Sample Clustering Plot](../supplementary/plots/quiz_01_q12_plot.png)

12. Using the generated plot below, which of the following statistical learning tasks does this data representation most likely relate to?

A. Classification  
B. Regression  
C. Clustering  
D. Dimensionality Reduction  

13. What is the main reason why observations in data science are often modeled as a combination of signal and random noise?

A. To oversimplify data analysis  
B. To account for variability in data  
C. To ignore the influence of random factors  
D. To only consider deterministic factors  

14. What are the correct formulations of the following aspects in supervised learning tasks? Select all that apply.

A. $Y_i = f(X_i) + \beta$  
B. $\bar{x} = \frac{1}{N}\sum_{i=1}^N x_i$  
C. $s^2 = \frac{1}{N-1}\sum_{i=1}^N(x_i - \bar{x})^2$  
D. $Y_i = f(X_i) + \epsilon_i$  

15. What will the following code snippet likely produce regarding model complexity in context of overfitting and underfitting? 

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Create some data
data = np.random.randn(100, 1)
target = 2 * data.flatten() + 1 + np.random.randn(100) * 0.1

X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)
plt.scatter(X_test, y_test)
plt.plot(X_test, model.predict(X_test), color='red')
plt.show()
```
Based on the outcome of the prediction line, what concept might be illustrated?

A. Overfitting as the model fits the training data too closely  
B. Underfitting as the model fails to capture the complexity of the data  
C. The importance of randomness in data  
D. The effectiveness of regression techniques  


# Quiz Generation Audit

## Generation Summary

- **Generated**: 2026-09-01 11:36:37
- **Input Source**: Notebook-supplied files
- **Output Directory**: /Users/jonathanma/Desktop/Projects/quizzer/outputs/notebook_run

## Request Parameters

- **Quiz Versions Requested**: 1
- **Questions per Version**: 15
- **Question Type**: mixed
- **Output Format**: markdown
- **Topic Filters**: None
- **Multiple-select Questions per Version**: 5

## Source Materials

**Materials Included in Generation:**

- MLO/01_Desc.md
- lecture_notes/01-Desc.html
- lecture_notes/01-Intro.pdf
- syllabus/syllabus.md

## Verification Results

- **Versions Generated**: 1 (Expected: 1) ✓
- **Answer Keys Generated**: 1 ✓
- **Quiz Output Files**: True ✓
- **Answer Key Output Files**: True ✓
- **Supplementary Files**: True ✓
- **Blueprint Parity**: PASS (all versions generated from the same assessment slots)

## Assessment Blueprint

| # | Topic | Learning outcome | Type | Modality | Difficulty |
|---:|---|---|---|---|---|
| 1 | Desc | LO-1: Python and Jupyter Notebook support interactive data analysis with code, text, equations, and visualizations. | single_choice | conceptual | foundational |
| 2 | Intro | LO-2: Pandas loads and manipulates tabular data; NumPy supports numerical arrays; Matplotlib supports visualization. | multiple_select | formula | intermediate |
| 3 | Desc | LO-3: Data can be loaded from CSV files, accessed by column name or position, and described using `shape` and `type`. | single_choice | code | intermediate |
| 4 | Intro | LO-4: Data science often models observations as structured signal plus random noise, e.g. $Y_i=f(X_i)+\epsilon_i$. | single_choice | plot_interpretation | advanced |
| 5 | Desc | LO-5: Statistical learning includes supervised tasks such as classification and regression, and unsupervised tasks such as clustering and dimensionality reduction. | multiple_select | conceptual | foundational |
| 6 | Intro | LO-6: Model complexity affects performance: simple models may underfit, while overly complex models may overfit training data. | single_choice | formula | intermediate |
| 7 | Desc | LO-7: Descriptive statistics summarize data using measures of location, dispersion, and shape. | single_choice | code | intermediate |
| 8 | Intro | LO-8: Location includes mode, mean, and median; dispersion includes variance and standard deviation: | multiple_select | plot_interpretation | advanced |
| 9 | Desc | LO-9: Shape can be described using sample skew and excess kurtosis; KDE approximates a distribution from data using a kernel and bandwidth. | single_choice | conceptual | foundational |
| 10 | Intro | LO-10: define mean, median, mode, variance, standard deviation, skew, excess kurtosis, kernel, and bandwidth. | single_choice | formula | intermediate |
| 11 | Desc | LO-11: identify Pandas, NumPy, and Matplotlib as tools for data manipulation, numerical computing, and visualization. | multiple_select | code | intermediate |
| 12 | Intro | LO-12: identify classification, regression, clustering, and dimensionality reduction as common statistical learning tasks. | single_choice | plot_interpretation | advanced |
| 13 | Desc | LO-13: explain observations as a combination of signal and random noise. | single_choice | conceptual | foundational |
| 14 | Intro | LO-14: distinguish supervised from unsupervised learning and discrete from continuous learning tasks. | multiple_select | formula | intermediate |
| 15 | Desc | LO-15: explain how model complexity relates to underfitting and overfitting. | single_choice | code | intermediate |

## Output Files Generated

**Quiz Files:**

- notebook_run/quizzes/quiz_01.md

**Answer Key Files:**

- notebook_run/answer_keys/quiz_01_key.md

**Supplementary Files:**

- notebook_run/supplementary/code/quiz_01_q04_plot.py
- notebook_run/supplementary/plots/quiz_01_q04_plot.png
- notebook_run/supplementary/code/quiz_01_q08_plot.py
- notebook_run/supplementary/plots/quiz_01_q08_plot.png
- notebook_run/supplementary/code/quiz_01_q12_plot.py
- notebook_run/supplementary/plots/quiz_01_q12_plot.png